# Usage Profiles: k-means Clustering

Segmentation of service users by how they engage with the service over time, using engagement features derived from a session log.

**All data in this notebook are simulated.** Feature distributions, sample size, the chosen number of clusters and the profile labels are invented for demonstration and do not correspond to any real dataset.

### Imports

In [ ]:
import numpy as np                      # numerical arrays and random numbers
import pandas as pd                     # tabular data (DataFrames)
import matplotlib.pyplot as plt         # plotting
import seaborn as sns                   # statistical plots on top of matplotlib
from sklearn.cluster import KMeans                  # clustering algorithm
from sklearn.decomposition import PCA               # dimensionality reduction (used only for plotting)
from sklearn.impute import SimpleImputer            # fills any remaining missing values
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score  # cluster-quality indices
from sklearn.preprocessing import StandardScaler    # standardises features to mean 0, sd 1

### Simulated session log and feature engineering

Each simulated user has a latent usage style (number of active months, spread, intensity). Their sessions are aggregated into monthly engagement features:

| Feature | Description |
| --- | --- |
| `active_months` | Distinct months with at least one session |
| `total_months` | Span in months between first and last session |
| `sessions_per_month` | Mean sessions in active months |
| `sessions_in_peak_month` | Maximum sessions in a single month |
| `gini_monthly` | Gini coefficient of the monthly distribution (unevenness of use); set to 0 for single-month users |

In [ ]:
# Random generator with a fixed seed so results are reproducible
rng = np.random.default_rng(11)
N_USERS = 5000

# Assign each user a hidden usage style (0-3) with these probabilities
# 0 = one-off, 1 = short and intensive, 2 = moderate, 3 = long-term
styles = rng.choice(4, N_USERS, p=[0.4, 0.25, 0.2, 0.15])

# Number of months in which each user is active, depending on their style
# (rng.integers excludes the upper bound: integers(2, 4) gives 2 or 3)
n_active = np.select([styles == 0, styles == 1, styles == 2, styles == 3],
                     [1, rng.integers(2, 4, N_USERS), rng.integers(3, 8, N_USERS), rng.integers(6, 13, N_USERS)])

# Extra calendar months over which the active months are scattered
# (0 = consecutive months, larger = use spread over a longer period)
spread = np.select([styles == 0, styles == 1, styles == 2, styles == 3],
                   [0, rng.integers(0, 3, N_USERS), rng.integers(2, 12, N_USERS), rng.integers(8, 24, N_USERS)])

# Average number of sessions per active month for each style
intensity = np.select([styles == 0, styles == 1, styles == 2, styles == 3], [1.3, 3.0, 1.8, 1.5])

# Build a monthly log: one row per user per active month
rows = []
for uid in range(N_USERS):
    # Choose which months the user was active: n_active distinct months
    # drawn from a window of n_active + spread months, sorted in time
    months = np.sort(rng.choice(max(n_active[uid], n_active[uid] + spread[uid]), n_active[uid], replace=False))
    # Sessions per active month: at least 1, plus a Poisson count, so the mean equals `intensity`
    counts = 1 + rng.poisson(intensity[uid] - 1, n_active[uid]) if intensity[uid] > 1 else np.ones(n_active[uid], int)
    rows.append(pd.DataFrame({"user_id": uid, "month": months, "n_sessions": counts}))
monthly = pd.concat(rows, ignore_index=True)   # stack all users into one long table


def gini(x):
    """Gini coefficient of a user's monthly session counts.

    0 = sessions spread perfectly evenly across months; values closer to 1 =
    sessions concentrated in a few months. A user with a single active month
    has no variation in use, so their Gini is set to 0.
    """
    x = np.sort(np.asarray(x, dtype=float))   # sort counts in ascending order
    n = len(x)                                # number of active months
    if n < 2 or x.sum() == 0:
        return 0.0                            # single-month users: no unevenness
    # Standard formula on sorted values: sum((2i - n - 1) * x_i) / (n * sum(x))
    return (2 * np.arange(1, n + 1) - n - 1).dot(x) / (n * x.sum())


# Collapse the monthly log into one row per user with five engagement features
features = monthly.groupby("user_id").agg(
    active_months=("month", "nunique"),                          # distinct active months
    total_months=("month", lambda m: m.max() - m.min() + 1),     # span from first to last month (inclusive)
    sessions_per_month=("n_sessions", "mean"),                   # average intensity
    sessions_in_peak_month=("n_sessions", "max"),                # busiest month
    gini_monthly=("n_sessions", gini),                           # unevenness of use over time
)

# Summary statistics (count, mean, sd, quartiles) for each feature
features.describe().round(2)

### Preprocessing

Any remaining missing values are imputed with the median (a safeguard: after setting single-month Ginis to 0 there should be none), and features are standardised so that each one contributes equally to the distances used by k-means.

In [ ]:
FEATURES = list(features.columns)

# Fill any missing values with the column median (k-means cannot handle NaN)
df_imputed = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(features), columns=FEATURES)

# Convert every feature to z-scores: (x - mean) / sd.
# k-means uses Euclidean distance, so without scaling the features with large
# ranges (e.g. total_months) would dominate those with small ranges (e.g. Gini)
X_scaled = StandardScaler().fit_transform(df_imputed)

### Model selection

k-means is fitted for k = 2 to 8 (the largest number still considered interpretable) and each solution is scored with inertia (elbow), silhouette, Calinski–Harabasz and Davies–Bouldin.

In [ ]:
rows = []
for k in range(2, 9):
    # n_init=10: run k-means from 10 different random starts and keep the best,
    # to avoid getting stuck in a poor local solution
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)   # fit the model and get each user's cluster
    rows.append({
        "k": k,
        # Inertia: within-cluster sum of squared distances (lower = tighter;
        # always decreases with k, so we look for the "elbow")
        "inertia": km.inertia_,
        # Silhouette: how much closer points are to their own cluster than to
        # the nearest other cluster, from -1 to 1 (higher = better)
        "silhouette": silhouette_score(X_scaled, labels),
        # Calinski-Harabasz: between-cluster vs within-cluster variance (higher = better)
        "calinski_harabasz": calinski_harabasz_score(X_scaled, labels),
        # Davies-Bouldin: average similarity of each cluster to its most similar one (lower = better)
        "davies_bouldin": davies_bouldin_score(X_scaled, labels),
    })
results = pd.DataFrame(rows).set_index("k")

# Elbow plot and silhouette plot side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(results.index, results["inertia"], "o-")
axes[0].set(title="Elbow method", xlabel="k", ylabel="Inertia (SSE)")
axes[1].plot(results.index, results["silhouette"], "o-", color="tab:red")
axes[1].set(title="Silhouette score", xlabel="k", ylabel="Silhouette")
plt.tight_layout()
plt.show()

results.round(3)

The indices are weighed together, giving priority to the silhouette and Calinski–Harabasz scores. Here the choice is automated with a simple rank average:

In [ ]:
# Rank each k on every index (rank 1 = best), respecting the direction of each index
ranks = pd.concat([
    results["silhouette"].rank(ascending=False),         # higher is better
    results["calinski_harabasz"].rank(ascending=False),  # higher is better
    results["davies_bouldin"].rank(),                    # lower is better
], axis=1)

# Silhouette and Calinski-Harabasz count double
weights = np.array([2, 2, 1])

# Weighted sum of ranks per k; the k with the lowest total is selected
best_k = int((ranks * weights).sum(axis=1).idxmin())
print("Selected k:", best_k)

### Cluster description

In [ ]:
# Fit the final model with the selected number of clusters
final = KMeans(n_clusters=best_k, random_state=42, n_init=10)

# Labels are attached to the unscaled table so profiles can be read in real units
df_imputed["cluster"] = final.fit_predict(X_scaled)

# Mean value of each feature per cluster, plus the number of users in each
summary = df_imputed.groupby("cluster")[FEATURES].mean()
summary["n_users"] = df_imputed["cluster"].value_counts().sort_index()
summary.round(2)

Each cluster gets a descriptive label from its feature profile (a simple rule here, standing in for a manual or LLM-assisted naming step):

In [ ]:
def label_cluster(row):
    """Name a cluster from its mean feature profile."""
    if row["active_months"] < 1.5:
        return "One-off"
    if row["sessions_per_month"] > 2.5:
        return "Intensive short-term"
    if row["total_months"] > 12:
        return "Long-term"
    return "Regular moderate" if row["gini_monthly"] < 0.15 else "Irregular moderate"


# Apply the rule to each cluster's profile
names = summary.apply(label_cluster, axis=1)

# If two clusters get the same name, number the duplicates: "Long-term (2)", ...
names = names + names.groupby(names).cumcount().map(lambda i: "" if i == 0 else f" ({i + 1})")

# Attach the profile name to every user
df_imputed["profile"] = df_imputed["cluster"].map(names)
names

In [ ]:
# Project the 5 standardised features onto 2 principal components for plotting only
# (the clustering itself used all 5 features)
pcs = PCA(n_components=2).fit_transform(X_scaled)
plot_df = pd.DataFrame(pcs, columns=["PC1", "PC2"]).assign(profile=df_imputed["profile"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: users in PCA space coloured by profile (random sample of 2,000 to avoid overplotting)
sns.scatterplot(data=plot_df.sample(2000, random_state=0), x="PC1", y="PC2", hue="profile",
                palette="tab10", s=12, ax=axes[0])
axes[0].set_title("Clusters in PCA space")

# Right: number of users per profile, largest first
order = df_imputed["profile"].value_counts().index
sns.countplot(data=df_imputed, x="profile", order=order, hue="profile", palette="tab10", legend=False, ax=axes[1])
axes[1].set_title("Users per profile")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()